# DMRE Re-ranker: Training Notebook



## 1. Install and import dependencies

In [1]:
# Run this cell once if packages are missing
# !pip install xgboost scikit-learn pandas numpy joblib

In [2]:
!pip install xgboost

In [3]:
!pip install joblib

In [4]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb

np.random.seed(42)  # for reproducibility
print('Libraries loaded successfully.')

Libraries loaded successfully.


## 2. Generate the synthetic dataset

Each row represents one (query, candidate_memory) pair.

**Features (what the model sees):**

| Feature | Meaning |
|---|---|
| `semantic_similarity` | Cosine similarity from ChromaDB (0–1) |
| `recency_days` | Days since the memory was created |
| `engagement_score` | Composite of visit count and dwell time (0–100) |
| `term_overlap` | Jaccard similarity between query and page title |
| `time_of_day_match` | Alignment between query hour and visit hour |

**Target:** `relevance` — continuous score in [0, 1]

We simulate 200 queries with 20 candidates each = 4,000 rows total.

In [5]:
NUM_QUERIES = 200
CANDIDATES_PER_QUERY = 20

def generate_row(query_id, rank_in_query):
    # Higher-ranked candidates have higher similarity on average
    base = max(0.3, 0.95 - (rank_in_query * 0.03))
    semantic_similarity = np.clip(base + np.random.normal(0, 0.08), 0.0, 1.0)

    recency_days = min(np.random.exponential(scale=30), 365)
    engagement_score = np.random.beta(1.5, 5) * 100
    term_overlap = np.random.beta(2, 5)
    time_of_day_match = np.random.beta(2, 2)

    # Ground-truth relevance: weighted combination + noise
    relevance = (
        0.45 * semantic_similarity +
        0.20 * (1 - min(recency_days / 180, 1.0)) +
        0.15 * (engagement_score / 100) +
        0.15 * term_overlap +
        0.05 * time_of_day_match
    )
    relevance = np.clip(relevance + np.random.normal(0, 0.05), 0.0, 1.0)

    return {
        'query_id': query_id,
        'semantic_similarity': round(semantic_similarity, 4),
        'recency_days': round(recency_days, 2),
        'engagement_score': round(engagement_score, 2),
        'term_overlap': round(term_overlap, 4),
        'time_of_day_match': round(time_of_day_match, 4),
        'relevance': round(relevance, 4),
    }

rows = [generate_row(qid, rank)
        for qid in range(NUM_QUERIES)
        for rank in range(CANDIDATES_PER_QUERY)]

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Generated {len(df)} rows across {df.query_id.nunique()} queries.')
df.head()

Generated 4000 rows across 200 queries.


,query_id,semantic_similarity,recency_days,engagement_score,term_overlap,time_of_day_match,relevance
0,27,0.4461,12.03,56.52,0.3349,0.6677,0.5388
1,174,0.6188,45.68,49.44,0.0419,0.6573,0.4899
2,26,0.7317,16.12,37.05,0.1664,0.6853,0.5573
3,196,0.7263,11.90,38.78,0.4306,0.4495,0.6909
4,149,0.6845,33.46,46.62,0.1954,0.6846,0.5488


In [6]:
# Dataset summary statistics
df.describe()

,query_id,semantic_similarity,recency_days,engagement_score,term_overlap,time_of_day_match,relevance
count,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000
mean,99.500000,0.662326,29.950973,23.181182,0.287755,0.501818,0.566668
std,57.741523,0.187777,29.358496,15.427955,0.158209,0.221845,0.108749
min,0.000000,0.073100,0.000000,0.150000,0.005700,0.005400,0.152700
25%,49.750000,0.512950,8.717500,11.070000,0.164925,0.329300,0.490875
50%,99.500000,0.665550,21.045000,20.195000,0.265850,0.503900,0.567850
75%,149.250000,0.812400,42.707500,32.617500,0.390300,0.676425,0.645800
max,199.000000,1.000000,205.150000,79.990000,0.869000,0.994800,0.947800


In [7]:
# Save a copy of the dataset for reference
df.to_csv('reranker_dataset.csv', index=False)
print('Dataset saved to reranker_dataset.csv')

Dataset saved to reranker_dataset.csv


## 3. Train/test split — grouped by query

We use `GroupShuffleSplit` so that all 20 candidates of a given query stay together — either entirely in train or entirely in test. This prevents data leakage: the model must prove it can rank candidates for **queries it has never seen**.

In [8]:
FEATURES = ['semantic_similarity', 'recency_days', 'engagement_score',
            'term_overlap', 'time_of_day_match']
TARGET = 'relevance'

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df['query_id']))
df_train, df_test = df.iloc[train_idx], df.iloc[test_idx]

X_train, y_train = df_train[FEATURES], df_train[TARGET]
X_test,  y_test  = df_test[FEATURES],  df_test[TARGET]

print(f'Training set: {len(X_train)} rows ({df_train.query_id.nunique()} queries)')
print(f'Test set:     {len(X_test)} rows ({df_test.query_id.nunique()} queries)')

Training set: 3200 rows (160 queries)
Test set:     800 rows (40 queries)


## 4. Train the XGBoost re-ranker

In [9]:
model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    objective='reg:squarederror',
    random_state=42,
)
model.fit(X_train, y_train)
print('Training complete.')

Training complete.


## 5. Evaluate the model

Two families of metrics:
- **Regression** (MAE, RMSE): how close predicted scores are to true scores
- **Information retrieval** (NDCG@5, MRR): how well the model orders results
- **Train accuracy** (Train acc): The accuracy of the model on training
- **Test accuracy** (Test acc): The accuracy of the model on testing

In [10]:
def ndcg_at_k(y_true, y_pred, k=5):
    order = np.argsort(y_pred)[::-1][:k]
    gains = np.asarray(y_true)[order]
    discounts = 1.0 / np.log2(np.arange(2, len(gains) + 2))
    dcg = np.sum(gains * discounts)
    ideal = np.sort(y_true)[::-1][:k]
    idcg = np.sum(ideal * discounts[:len(ideal)])
    return dcg / idcg if idcg > 0 else 0.0

def mean_reciprocal_rank(y_trues, y_preds, threshold=0.6):
    rrs = []
    for y_true, y_pred in zip(y_trues, y_preds):
        order = np.argsort(y_pred)[::-1]
        found = False
        for rank, idx in enumerate(order, start=1):
            if y_true[idx] >= threshold:
                rrs.append(1.0 / rank)
                found = True
                break
        if not found:
            rrs.append(0.0)
    return float(np.mean(rrs))

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

df_eval = df_test.copy()
df_eval['predicted'] = y_pred
ndcgs, y_trues, y_preds_list = [], [], []
for _, g in df_eval.groupby('query_id'):
    ndcgs.append(ndcg_at_k(g[TARGET].values, g['predicted'].values, k=5))
    y_trues.append(g[TARGET].values)
    y_preds_list.append(g['predicted'].values)

print('=== Regression metrics ===')
print(f'MAE:    {mae:.4f}')
print(f'RMSE:   {rmse:.4f}')
print()
print(f'Train acc : {model.score(X_train, y_train):.3f}')
print(f'Test acc : {model.score(X_test, y_test):.3f}')
print()
print('=== Information retrieval metrics ===')
print(f'NDCG@5: {np.mean(ndcgs):.4f}')
print(f'MRR:    {mean_reciprocal_rank(y_trues, y_preds_list):.4f}')

=== Regression metrics ===
MAE:    0.0424
RMSE:   0.0538

Train acc : 0.914
Test acc : 0.754

=== Information retrieval metrics ===
NDCG@5: 0.9749
MRR:    0.9875


In [11]:
# Feature importance — which signals matter most?
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print('Feature importance:')
for name, score in importances.items():
    print(f'  {name:<22s} {score:.4f}')

Feature importance:
  semantic_similarity    0.7142
  recency_days           0.1057
  term_overlap           0.0772
  engagement_score       0.0656
  time_of_day_match      0.0373


## 6. Save the trained model

The `.pkl` file is what your FastAPI backend will load at startup.

In [12]:
joblib.dump(model, 'reranker_model.pkl')
print('Model saved to reranker_model.pkl')

Model saved to reranker_model.pkl


## 7. Demonstration: re-ranking real candidates

This simulates what happens in production. Given a user query and a list of candidates returned by ChromaDB, the model produces a score for each and reorders them. The top-ranked result is the one shown first in the dashboard.

In [13]:
from datetime import datetime, timedelta

def rerank(query, candidates, top_k=5):
    """candidates: list of dicts with keys matching FEATURES."""
    X = pd.DataFrame(candidates)[FEATURES]
    scores = model.predict(X)
    ranked = sorted(zip(candidates, scores), key=lambda p: p[1], reverse=True)
    return ranked[:top_k]

# Example: user searches "react hooks"
candidates = [
    {'title': 'React hooks tutorial',      'semantic_similarity': 0.92,
     'recency_days': 3,   'engagement_score': 75, 'term_overlap': 0.6, 'time_of_day_match': 0.9},
    {'title': 'Python basics',             'semantic_similarity': 0.75,
     'recency_days': 100, 'engagement_score': 10, 'term_overlap': 0.0, 'time_of_day_match': 0.3},
    {'title': 'Advanced hooks patterns',   'semantic_similarity': 0.88,
     'recency_days': 10,  'engagement_score': 55, 'term_overlap': 0.5, 'time_of_day_match': 0.7},
    {'title': 'CSS grid guide',            'semantic_similarity': 0.65,
     'recency_days': 30,  'engagement_score': 30, 'term_overlap': 0.0, 'time_of_day_match': 0.5},
]

print('Query: "react hooks"')
print('Re-ranked results:')
for i, (cand, score) in enumerate(rerank('react hooks', candidates), 1):
    print(f'  {i}. [{score:.3f}]  {cand["title"]}')

Query: "react hooks"
Re-ranked results:
  1. [0.847]  React hooks tutorial
  2. [0.763]  Advanced hooks patterns
  3. [0.521]  CSS grid guide
  4. [0.464]  Python basics


## 8. Summary

- Trained an XGBoost re-ranker on 4,000 synthetic (query, candidate) pairs
- 80/20 train-test split grouped by query, preventing data leakage
- Achieved strong ranking metrics on unseen queries
- Model saved as `reranker_model.pkl` for production loading in FastAPI

The trained model integrates into the DMRE backend as the second stage of a two-stage retrieval pipeline: Sentence-BERT + ChromaDB for candidate retrieval, then this XGBoost model for final ordering.